# Memoization, Tabulation, and Repeated Work


```{contents}
:local:
:depth: 2
```


```{index} Save Answers to Repeated Questions, Not Arbitrary Intermediate State
```



```{index} Compare Three Executions of the Same Dependency Rule
```

## Compare Three Executions of the Same Dependency Rule


A repeated subproblem is a question with the same complete state, not just a similarly named function call. With F(0)=0, F(1)=1 and F(n)=F(n-1)+F(n-2), the integer n fully identifies the question. Fix that definition before sharing results. Changing the base values creates a different sequence and invalidates an old cache.

For F(4), the naive call tree asks for F(3) and F(2). F(3) asks for F(2) and F(1), so F(2) is expanded twice. Termination is already guaranteed by decreasing n; the problem is repeated work.

```text
F(4)
  F(3)
    F(2)
      F(1)
      F(0)
    F(1)
  F(2)
    F(1)
    F(0)
```

There are nine invocations including base cases. Only four calls are simultaneously active on the deepest path. A cache can remove repeated expansion without turning the recursion into constant-depth execution.


In [1]:
using System;
using System.Collections.Generic;
var cache = new Dictionary<int, long>();
int calls = 0, hits = 0, computed = 0;
long Fib(int n)
{
    calls++;
    if (n <= 1) return n;
    if (cache.TryGetValue(n, out long saved))
    {
        hits++;
        Console.WriteLine($"Hit F({n})={saved}");
        return saved;
    }
    long value = checked(Fib(n - 1) + Fib(n - 2));
    cache[n] = value;
    computed++;
    Console.WriteLine($"Store F({n})={value}");
    return value;
}
long result = Fib(4);
Console.WriteLine($"F(4)={result}; calls={calls}; hits={hits}; computed={computed}");


Store F(2)=1
Store F(3)=2
Hit F(2)=1
Store F(4)=3
F(4)=3; calls=7; hits=1; computed=3


The cell intentionally calls only a fixed valid argument. The complete APIs below retain domain and resource limits. Base cases are returned directly rather than stored; therefore computed counts non-base cache misses, not every call. Cache F(2) only after both dependencies return, otherwise another call could read an incomplete result.

| Event during F(4) | Completed cache afterward | Significance |
| --- | --- | --- |
| Finish F(2) within F(3) | 2→1 | First non-base state completed |
| Finish F(3) | 2→1, 3→2 | Its dependencies were available |
| Request F(2) from F(4) | Unchanged | Cache hit avoids another expansion |
| Finish F(4) | 2→1, 3→2, 4→3 | Requested result is now complete |

For n≥2, there are n-1 distinct non-base states. Each is expanded once and makes two recursive calls, so total invocations are the initial call plus twice the number of expansions: 2n-1. For n=4 this is seven. Exact counts depend on this implementation's decision not to cache base cases; the linear growth claim does not depend on adopting this exact convention.


**Common mistake.** Keying the cache on something other than the complete state. A cache keyed only by `n` is correct here because `n` is the whole question; reusing it after changing the base values, or for a two-parameter problem, returns answers to a different question.


```{index} Fill the Same States Without Recursive Calls
```

## Fill the Same States Without Recursive Calls


Tabulation stores the same mathematical answers but selects an order explicitly. For increasing i, both i-1 and i-2 already exist. Initialization supplies F(0) and F(1); it is not an incidental convenience. The small cell below makes every completed table entry visible.


In [1]:
using System;
int n = 4;
long[] table = new long[n + 1];
table[0] = 0;
if (n >= 1) table[1] = 1;
for (int i = 2; i <= n; i++)
{
    table[i] = checked(table[i - 1] + table[i - 2]);
    Console.WriteLine($"F({i})={table[i]}");
}
Console.WriteLine($"Table: {string.Join(",", table)}");


F(2)=1
F(3)=2
F(4)=3
Table: 0,1,1,2,3


Guided task: append the row for i=5, naming the two source entries and the result. Then describe why filling from n down to two would read entries that are still only default zero. Change n to zero and check why the guard on initialization of entry one is necessary.

```{dropdown} Check the dependency order
F(5)=F(4)+F(3)=3+2=5. Descending fill reaches F(5) before its non-base predecessors have been computed, so default zeros are not valid answers. When n is zero, the array contains only entry zero; writing entry one would be outside it.
```

For transfer, keep the recurrence but choose F(0)=2 and F(1)=1. Derive the first six values before adapting both strategies, and explain why the old cache must not be reused. The chapter homework uses another recurrence; first identify its states, base cases, dependency order and requested output. Rolling storage is appropriate when only the final value is needed, while a full table supports inspection of all earlier entries.


**Common mistake.** Filling the table from `n` downward. Entry `i` needs `i - 1` and `i - 2`, which are still default zeros when the loop runs backward, so the table silently fills with wrong values and no exception signals it.


```{index} Save Answers to Repeated Questions, Not Arbitrary Intermediate State
```

## Save Answers to Repeated Questions, Not Arbitrary Intermediate State



Fibonacci gives a small example where termination alone does not prevent repeated work. Computing F(5) asks for F(4) and F(3); computing F(4) asks for F(3) again. A cache can retain the completed answer for a numeric argument and reuse it when that same subproblem returns. The key is sufficient because the recurrence and base cases are fixed for this computation.

Memoization starts from the requested state and recursively solves missing dependencies. Tabulation chooses an order in which needed dependencies have already been solved. For Fibonacci, increasing argument order works because F(i) depends on smaller arguments. These are two execution strategies for a stated dependency structure, not two unrelated tricks for making code faster.

| i | F(i) | Dependencies already available |
| --- | --- | --- |
| 0 | 0 | Base case |
| 1 | 1 | Base case |
| 2 | 1 | F(1), F(0) |
| 3 | 2 | F(2), F(1) |
| 4 | 3 | F(3), F(2) |
| 5 | 5 | F(4), F(3) |

The table invariant is that every earlier stored entry has its correct specified value before the next entry is computed. Storing a result before all dependencies have been completed can cache a partial value as if it were final. Reusing one global cache across requests with different policies can likewise mix different problems under the same key.

Rolling storage keeps only the two predecessor values when only the final Fibonacci result is required. It cannot simultaneously promise to return all earlier table entries without retaining or recomputing them. This is a tradeoff between requested output and retained state. Memoization also does not eliminate recursive stack depth simply because repeated arithmetic has been removed.

Guided exercise: derive F(6) from the table, then sketch which repeated calls a naive recursion would make. Label cache misses and cache hits separately from active call frames. Finally, change the base values and explain why a cache produced under the previous definition cannot be reused without identifying which definition its keys belong to.

For related explanation, see the further reading below.[^development-reading]


```{index} Memoization and Tabulation
```

## Memoization and Tabulation



A **state** identifies one smaller problem completely: for Fibonacci the integer n, for the purchasing plan of the next section the remaining amount, for a one-use inventory the position reached and the amount still needed. A **transition** relates a state to the states it depends on. Dynamic programming is the discipline of writing those two things down before writing code, and then choosing one of two execution strategies for the same dependency structure.

| Step | Question to answer | Fibonacci |
| --- | --- | --- |
| State | What information identifies a subproblem? | n |
| Base cases | Which states are answered directly? | F(0) = 0, F(1) = 1 |
| Transition | How does a state combine its dependencies? | F(n) = F(n - 1) + F(n - 2) |
| Order | In which sequence are dependencies ready? | Increasing n |
| Result | Which state, or which reconstruction, is requested? | F(n), or the whole table |

**Memoization** keeps the recursive shape of the transition and adds a cache: a state is computed the first time it is requested and looked up afterward. It visits only the states the requested answer needs, which can be far fewer than all of them, and it keeps the recursion's stack depth. **Tabulation** replaces the recursion with a loop that fills states in dependency order, so every dependency is already in the table when it is read. It has no stack depth, usually a smaller constant factor, and the chance to keep only the last few rows when the full table is not requested; it may compute states the answer never uses. Both give the same values for the same definition, which is why the next two subsections run both on Fibonacci and count their work.

Counting problems such as Fibonacci need only a decomposition. Optimization problems such as the minimum-pack plan additionally need **optimal substructure**: an optimal answer for a state must be built from optimal answers for its dependencies, or the transition's minimum is not justified. The next section states that requirement precisely.

```{index} Recognize a Dynamic Programming Problem
```

### Recognize a Dynamic Programming Problem

Two properties together signal that dynamic programming will pay off. **Overlapping subproblems**: the straightforward recursion asks the same question many times, as Fibonacci asks for F(2) repeatedly. Without overlap, a cache stores every answer once and reads none of them back, and plain divide-and-conquer is the right tool; merge sort has no repeated subproblems and gains nothing from a table. **Optimal substructure**, for optimization problems: the best answer for a state can be assembled from the best answers for smaller states. Without it, the minimum over transitions is not the true minimum, and the table is wrong even when it is filled correctly.

| Problem | State | Overlap? | Optimal substructure? | Method |
| --- | --- | --- | --- | --- |
| Fibonacci numbers | n | Yes | Not needed (counting) | Memoize or tabulate |
| Fewest packs for an amount, unlimited packs | Amount | Yes | Yes | Tabulate by amount (25.2) |
| Fewest packs with each pack usable once | Position and amount | Yes | Yes | Two-dimensional table or memoized search (25.3) |
| Shortest path in a graph with cycles, nonnegative weights | Vertex | Yes | Yes | Dijkstra is the greedy specialization (24.3) |
| Sort an array | Range | No | Yes | Divide-and-conquer; no table |
| Maximum number of compatible meetings | Position in finish order | Yes | Yes, and a greedy choice is safe | Greedy is cheaper (24.1) |

The last row is the reminder that dynamic programming is not the first resort. When an exchange argument justifies a greedy choice, the table is unnecessary work; when no such argument exists, the table considers every last choice and is justified by optimal substructure. The purchasing plan in the next section sits exactly at that boundary: greedy fails on the 1, 3, 4 pack sizes, and the amount-indexed table repairs it.

```{index} Repeated Work Is Visible
```

## Repeated Work Is Visible




```{index} Repeated Work Is Visible
```

F(0)=0, F(1)=1, and F(n)=F(n−1)+F(n−2) for n≥2. The argument strictly decreases, but the recursion tree repeats arguments. Count every invocation, including base cases; computed counts only non-base expansions. Compare the same n, not unrelated examples.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

var naive=FibPlanning.Fibonacci(8,false);
var memo=FibPlanning.Fibonacci(8);
Console.WriteLine($"naive: value={naive.Value}, calls={naive.Calls}, computed={naive.Computed}");
Console.WriteLine($"memo: value={memo.Value}, calls={memo.Calls}, hits={memo.Hits}, computed={memo.Computed}");
public static class FibPlanning
{
    public record FibResult(long Value,int Calls,int Hits,int Computed);
    // F(0)=0, F(1)=1. F(92) is the last value representable by long.
    public static FibResult Fibonacci(int n,bool memoized=true)
    {
        if(n<0||n>92||(!memoized&&n>25))throw new ArgumentOutOfRangeException(nameof(n));
        var cache=new Dictionary<int,long>();int calls=0,hits=0,computed=0;
        long Solve(int k)
        {
            calls++;
            if(k<=1)return k;
            if(memoized&&cache.TryGetValue(k,out long saved)){hits++;return saved;}
            long value=checked(Solve(k-1)+Solve(k-2));computed++;
            if(memoized)cache[k]=value;
            return value;
        }
        return new FibResult(Solve(n),calls,hits,computed);
    }
    public static long[] FibonacciTable(int n)
    {
        if(n<0||n>92)throw new ArgumentOutOfRangeException(nameof(n));
        var table=new long[n+1];if(n>=1)table[1]=1;
        for(int i=2;i<=n;i++)table[i]=checked(table[i-1]+table[i-2]);
        return table;
    }
    public static long FibonacciRolling(int n)
    {
        if(n<0||n>92)throw new ArgumentOutOfRangeException(nameof(n));
        if(n==0)return 0;
        long before=0,current=1;
        for(int i=2;i<=n;i++)(before,current)=(current,checked(before+current));
        return current;
    }
}


naive: value=21, calls=67, computed=33
memo: value=21, calls=15, hits=5, computed=7


The naive call recurrence is C(0)=C(1)=1 and C(n)=1+C(n−1)+C(n−2); C(n)=2F(n+1)−1. Calls are exponential in n while active stack depth is Θ(n). This teaching implementation caps naive n at 25 to keep browser runs short; all Fibonacci APIs otherwise accept 0..92, the range that fits signed long. Invalid n is rejected before recursion.

**Common mistake.** Comparing the naive call count at one `n` with the memoized count at another. The two counts are only comparable for the same `n`; the cell runs both at 8 so that the difference is the repeated work alone.


```{index} Memoization Stores Complete States
```

## Memoization Stores Complete States




```{index} Memoization Stores Complete States
```

The cache key n fully identifies F(n) because the recurrence and base cases are fixed. Every cached value is a completed correct result. Compute a missing state recursively, then store it. Local cache ownership avoids mixing unrelated requests or caller-supplied incorrect entries.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

foreach(int n in new[]{0,1,2,8,40})
{
 var result=FibPlanning.Fibonacci(n);
 Console.WriteLine($"n={n}, value={result.Value}, calls={result.Calls}, hits={result.Hits}, computed={result.Computed}");
}
public static class FibPlanning
{
    public record FibResult(long Value,int Calls,int Hits,int Computed);
    // F(0)=0, F(1)=1. F(92) is the last value representable by long.
    public static FibResult Fibonacci(int n,bool memoized=true)
    {
        if(n<0||n>92||(!memoized&&n>25))throw new ArgumentOutOfRangeException(nameof(n));
        var cache=new Dictionary<int,long>();int calls=0,hits=0,computed=0;
        long Solve(int k)
        {
            calls++;
            if(k<=1)return k;
            if(memoized&&cache.TryGetValue(k,out long saved)){hits++;return saved;}
            long value=checked(Solve(k-1)+Solve(k-2));computed++;
            if(memoized)cache[k]=value;
            return value;
        }
        return new FibResult(Solve(n),calls,hits,computed);
    }
    public static long[] FibonacciTable(int n)
    {
        if(n<0||n>92)throw new ArgumentOutOfRangeException(nameof(n));
        var table=new long[n+1];if(n>=1)table[1]=1;
        for(int i=2;i<=n;i++)table[i]=checked(table[i-1]+table[i-2]);
        return table;
    }
    public static long FibonacciRolling(int n)
    {
        if(n<0||n>92)throw new ArgumentOutOfRangeException(nameof(n));
        if(n==0)return 0;
        long before=0,current=1;
        for(int i=2;i<=n;i++)(before,current)=(current,checked(before+current));
        return current;
    }
}


n=0, value=0, calls=1, hits=0, computed=0
n=1, value=1, calls=1, hits=0, computed=0
n=2, value=1, calls=3, hits=0, computed=1
n=8, value=21, calls=15, hits=5, computed=7
n=40, value=102334155, calls=79, hits=37, computed=39


For n≥2 there are n−1 distinct non-base states, 2n−1 calls including cache hits, and max(0,n−3) hits. Base results are not cached here. Expected constant-time dictionary lookup gives Θ(n) time, Θ(n) cache storage, and Θ(n) stack space. These are distinct costs. A fresh invocation starts a fresh cache; memoization does not make the recursive stack disappear. Induction proves correctness: bases are correct, recursive dependencies are smaller, and their sum is F(n).

```{index} Tabulation Follows Dependency Order
```

## Tabulation Follows Dependency Order




```{index} Tabulation Follows Dependency Order
```

The invariant before iteration i is that entries 0..i−1 equal their specified Fibonacci values. Writing table[i] from i−1 and i−2 preserves it. Initialize entry one only when it exists; zero must work.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

foreach(int n in new[]{0,1,10})
{
 var table=FibPlanning.FibonacciTable(n);
 Console.WriteLine($"n={n}: [{string.Join(",",table)}], rolling={FibPlanning.FibonacciRolling(n)}");
}
Console.WriteLine($"F(92)={FibPlanning.FibonacciRolling(92)}");
try{FibPlanning.FibonacciTable(93);}catch(ArgumentOutOfRangeException){Console.WriteLine("overflow domain rejected");}
public static class FibPlanning
{
    public record FibResult(long Value,int Calls,int Hits,int Computed);
    // F(0)=0, F(1)=1. F(92) is the last value representable by long.
    public static FibResult Fibonacci(int n,bool memoized=true)
    {
        if(n<0||n>92||(!memoized&&n>25))throw new ArgumentOutOfRangeException(nameof(n));
        var cache=new Dictionary<int,long>();int calls=0,hits=0,computed=0;
        long Solve(int k)
        {
            calls++;
            if(k<=1)return k;
            if(memoized&&cache.TryGetValue(k,out long saved)){hits++;return saved;}
            long value=checked(Solve(k-1)+Solve(k-2));computed++;
            if(memoized)cache[k]=value;
            return value;
        }
        return new FibResult(Solve(n),calls,hits,computed);
    }
    public static long[] FibonacciTable(int n)
    {
        if(n<0||n>92)throw new ArgumentOutOfRangeException(nameof(n));
        var table=new long[n+1];if(n>=1)table[1]=1;
        for(int i=2;i<=n;i++)table[i]=checked(table[i-1]+table[i-2]);
        return table;
    }
    public static long FibonacciRolling(int n)
    {
        if(n<0||n>92)throw new ArgumentOutOfRangeException(nameof(n));
        if(n==0)return 0;
        long before=0,current=1;
        for(int i=2;i<=n;i++)(before,current)=(current,checked(before+current));
        return current;
    }
}


n=0: [0], rolling=0
n=1: [0,1], rolling=1
n=10: [0,1,1,2,3,5,8,13,21,34,55], rolling=55
F(92)=7540113804746346429
overflow domain rejected


A full table costs Θ(n) time and Θ(n) storage without recursion. Rolling state keeps only the two needed predecessors, giving Θ(n) time and Θ(1) auxiliary storage when only the final value is requested. It cannot provide every historical table entry. Checked addition protects arithmetic; validating 0..92 also avoids computing an unnecessary F(93) on the final rolling update.

**Common mistake.** Writing `table[1] = 1` without the `n >= 1` guard. For `n = 0` the table has one entry, index 1 is out of range, and a method that is correct for every other input throws on the smallest one.


```{index} Code Tracing: Hits and Misses for F(6)
```

### Code Tracing: Hits and Misses for F(6)

The memoized `Fib` at the top of this page prints `Store` when it computes a non-base state and `Hit` when it reads one back. Predict the exact sequence of those lines for `Fib(6)`, in order, and the final counts of calls, hits, and computed states, using the formulas 2n - 1 calls and n - 3 hits from the earlier subsection. Then run the solution.

In [ ]:
// Solution
using System;
using System.Collections.Generic;

var cache = new Dictionary<int, long>();
int calls = 0, hits = 0, computed = 0;
long result = Fib(6);
Console.WriteLine($"F(6)={result}; calls={calls}; hits={hits}; computed={computed}");

long Fib(int n)
{
    calls++;
    if (n < 2) return n;
    if (cache.TryGetValue(n, out long known)) { hits++; Console.WriteLine($"Hit F({n})={known}"); return known; }
    long value = Fib(n - 1) + Fib(n - 2);
    cache[n] = value; computed++;
    Console.WriteLine($"Store F({n})={value}");
    return value;
}

Store F(2)=1
Store F(3)=2
Hit F(2)=1
Store F(4)=3
Hit F(3)=2
Store F(5)=5
Hit F(4)=3
Store F(6)=8
F(6)=8; calls=11; hits=3; computed=5


The stores arrive in increasing order, F(2) through F(6), because each state's first dependency is the state just below it and the recursion runs all the way down before anything completes. Each hit immediately follows the store of the state above it: after F(3) is stored, F(4) asks for F(2) and hits. There are 11 calls, 3 hits, and 5 computed states, matching 2n - 1, n - 3, and n - 1. A trace that lists a hit for F(1) has forgotten that base cases are returned directly and never cached in this implementation.

```{index} Code Completion: Count the Ways to Pay
```

### Code Completion: Count the Ways to Pay

A customer pays an amount in whole units using notes worth 1 or 2, and the order of the notes matters, so an amount of 3 can be paid as 1+1+1, 1+2, or 2+1. Complete `Ways` so that it is memoized: look the state up before computing it and store it after. The transition is `Ways(n) = Ways(n - 1) + Ways(n - 2)` with `Ways(0) = 1` and `Ways(1) = 1`. The expected output is `ways(3)=3`, `ways(10)=89`, `ways(60)=2504730781961`, and the final line must report far fewer than 2 to the 60th calls.

In [ ]:
using System;
using System.Collections.Generic;

var memo = new Dictionary<int, long>();
long calls = 0;
foreach (int n in new[] { 3, 10, 60 }) Console.WriteLine($"ways({n})={Ways(n)}");
Console.WriteLine($"calls={calls}");

long Ways(int n)
{
    calls++;
    if (n <= 1) return 1;
    // Your code starts here.
    return Ways(n - 1) + Ways(n - 2);
    // Your code ends here.
}

In [ ]:
// Solution
using System;
using System.Collections.Generic;

var memo = new Dictionary<int, long>();
long calls = 0;
foreach (int n in new[] { 3, 10, 60 }) Console.WriteLine($"ways({n})={Ways(n)}");
Console.WriteLine($"calls={calls}");

long Ways(int n)
{
    calls++;
    if (n <= 1) return 1;
    if (memo.TryGetValue(n, out long known)) return known;     // answered before: reuse
    long value = Ways(n - 1) + Ways(n - 2);
    memo[n] = value;                                           // store only a completed answer
    return value;
}

ways(3)=3
ways(10)=89
ways(60)=2504730781961
calls=121


The starter is correct and hopeless: without the cache, `Ways(60)` makes more than a quadrillion calls and never finishes in the browser. With the two added lines the three requests cost 121 calls in total, and the third request benefits from the states the second one already stored, because the memo outlives individual calls. The transition is Fibonacci's with different base cases, which is exactly why the cache keys must belong to this definition and not be shared with a Fibonacci cache.

```{index} Check Your Understanding
```

### Check Your Understanding

Answer each question before opening its dropdown.

1. A recursive method has no repeated subproblems. What does memoizing it cost and gain?

```{dropdown} Answer
It costs a dictionary insertion per call and a lookup that never hits, and gains nothing: every state is computed exactly once either way. Overlap is the precondition for memoization to help; merge sort is the standard example where it does not.
```

2. Why is storing a state in the cache before its dependencies have returned a defect even when the stored value is later overwritten?

```{dropdown} Answer
Another call for the same state can arrive while the first is still in progress, read the partial value, and build its own answer on it; by the time the correct value is stored, the wrong one has already been used. Store only after the state is complete.
```

3. Tabulated Fibonacci with rolling storage uses two variables. What question can it no longer answer?

```{dropdown} Answer
Any question about an earlier entry, such as F(k) for k < n, or the whole sequence. Rolling storage is right when only the final value is requested and wrong when the table itself is the output; the requested result decides the storage.
```

4. Two problems share the transition T(n) = T(n - 1) + T(n - 2) but differ in base cases. Can they share a cache?

```{dropdown} Answer
No. The cache maps a state to the answer under one definition; with different base cases every stored value differs. The cache must be owned by, or keyed with, the definition it was computed under.
```

```{rubric} Footnotes
```


[^development-reading]: Further reading: [OpenDSA, Dynamic Programming](https://opendsa.org/OpenDSA/Books/CS4104/html/DynamicProgramming.html). The worked business scenarios and guided reasoning here are original teaching examples.
